# 02 Event analysis

Flood events are separated in the observed and in each simulated series (NOAA-OWP hydrotools), matched one to one within 12 hours, and scored.

- **Detection**: POD = hits / (hits + misses), FAR = false alarms / (hits + false alarms), CSI = hits / (hits + misses + false alarms).
- **Matched events** (hits): peak ratio (simulated / observed), peak timing error, time to peak (rise time) error, volume error.
- **Event sets**: `all` = every detected event; `larger` = events at or above the 13th-largest observed peak of the gage (about 3 per year).

Hits and misses are counted in the season of the observed peak, false alarms in the season of the simulated peak.

**Paths.** Nothing to edit. The notebook finds the repository by itself and takes the data from the folder next to it (`cfe_evel/organized` and `cfe_evel/results`), so it runs the same from IDAS (`~/hpchome/Papers/cfe_evel/cfe_eval/analysis`) and from Argon. To use another data folder, set the environment variable `CFE_DATA_ROOT` before starting Jupyter.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# find the repository (the folder that holds cfe_eval/config.py), whatever folder the notebook is opened from
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cfe_eval" / "config.py").exists())
sys.path.insert(0, str(REPO))
from cfe_eval import common as cm
from cfe_eval import config as cfg
from cfe_eval import maps, plots

%matplotlib inline
plots.slide_style()
pd.set_option("display.max_columns", 60, "display.width", 220)

FIG = cfg.FIGURES / "02_events"            # figures of this notebook
FIG.mkdir(parents=True, exist_ok=True)
SAVE = True                             # write every figure as PNG, 300 dpi

def save(fig, name):
    if SAVE:
        fig.savefig(FIG / f"{name}.png")

print("repository:", REPO)
print("data root :", cfg.DATA_ROOT)
print("tables    :", cfg.TABLES, "(found)" if cfg.TABLES.exists() else "(MISSING: run the evaluation jobs first)")
print("figures   :", FIG)

## Settings

`USE_QAQC = True` keeps only the gages that passed the quality control (notebook 00, file `results/tables/qaqc_sites.csv`). If that file does not exist yet, all gages are used.

`EVENT_SET` chooses the events behind the per-gage figures. With `"larger"` a gage has only a few events per season, so the seasonal boxes become coarse; section 3 pools the larger floods instead.

In [ ]:
USE_QAQC = True
WHISKERS = 1.5      # box plot whiskers: 1.5 box heights; use (5, 95) for the 5th and 95th percentiles
EVENT_SET = "all"   # "all" or "larger"
keep = cm.qaqc_keep() if USE_QAQC else None

def load(name):
    """A results table, limited to the gages kept by the quality control."""
    df = cm.read_table(name)
    col = "site" if name == "gages" else "gage"
    return df if keep is None else df[df[col].isin(keep)].reset_index(drop=True)

gages = load("gages")
P = list(cfg.PRODUCTS)
print("quality control:", "not applied (all gages)" if keep is None else f"applied, {len(keep)} gages kept")
gages.groupby("aridity_zone").size().reindex(cfg.ZONES).rename("gages").to_frame().T

In [ ]:
ctg = load("contingency").merge(gages[["site", "lat", "lon"]].rename(columns={"site": "gage"}), on="gage")
ems = load("event_metrics_site").merge(gages[["site", "lat", "lon"]].rename(columns={"site": "gage"}), on="gage")
em = load("event_metrics")
print(f"{len(em):,} matched events, {em['gage'].nunique()} gages")

## 1. Overview

In [ ]:
def pooled(df, by):
    t = df.groupby(by)[["hits", "misses", "false_alarms"]].sum()
    t["POD"] = t["hits"] / (t["hits"] + t["misses"])
    t["FAR"] = t["false_alarms"] / (t["hits"] + t["false_alarms"])
    t["CSI"] = t["hits"] / (t["hits"] + t["misses"] + t["false_alarms"])
    t["bias"] = (t["hits"] + t["false_alarms"]) / (t["hits"] + t["misses"])
    return t

cf = ctg[ctg["period_type"] == "full"]
rows = {}
for eset in cfg.EVENT_SETS:
    d = cf[cf["event_set"] == eset]
    med = d.groupby("product")[["POD", "FAR", "CSI"]].median().add_prefix("median ")
    pl = pooled(d, "product")[["hits", "misses", "false_alarms", "POD", "FAR", "CSI"]]
    pl.columns = ["hits", "misses", "false alarms", "pooled POD", "pooled FAR", "pooled CSI"]
    rows[eset] = pd.concat([med, pl], axis=1).reindex(P)
overview = pd.concat(rows).round(2)
overview.to_csv(FIG / "detection_overview.csv")
overview

"median" = median over gages of the score of each gage; "pooled" = all events of all gages counted together.

Errors of the matched events, median over all matched events:

In [ ]:
cols = {"peak_ratio": "peak ratio (sim / obs)", "peak_err_pct": "peak error (%)",
        "time_to_peak_err_h": "peak timing error (h)", "rise_err_h": "time to peak error (h)",
        "vol_err_pct": "volume error (%)"}
ev_over = em.groupby(["event_set", "product"])[list(cols)].median().rename(columns=cols)
ev_over.insert(0, "matched events", em.groupby(["event_set", "product"]).size())
ev_over = ev_over.reindex([(s, p) for s in cfg.EVENT_SETS for p in P]).round(2)
ev_over.to_csv(FIG / "matched_events_overview.csv")
ev_over

## 2. Performance diagram

Each point pools the events of one climate type for one rain product. Right = fewer false alarms, up = fewer misses, grey curves = CSI, dashed lines = ratio of simulated to observed event counts (1 = as many simulated as observed events). The best corner is the top right (Roebber, 2009, https://doi.org/10.1175/2008WAF2222159.1).

In [ ]:
markers = {"Arid": "s", "Semi-arid": "^", "Semi-humid": "D", "Humid": "o"}
fig, axes = plt.subplots(1, 2, figsize=(20, 9.5))
for ax, eset in zip(axes, cfg.EVENT_SETS):
    plots.performance_diagram(ax)
    t = pooled(cf[cf["event_set"] == eset], ["aridity_zone", "product"])
    for z in cfg.ZONES:
        for p in P:
            r = t.loc[(z, p)]
            ax.scatter(1 - r["FAR"], r["POD"], s=260, marker=markers[z], color=plots.PRODUCT_COLORS[p][0],
                       edgecolor="black", linewidth=1.0, zorder=5)
    ax.set_title("All events" if eset == "all" else "Larger floods")
from matplotlib.lines import Line2D
h = [Line2D([], [], marker="o", linestyle="", markersize=14, markerfacecolor=plots.PRODUCT_COLORS[p][0],
            markeredgecolor="black", label=cfg.PRODUCT_LABEL[p]) for p in P]
h += [Line2D([], [], marker=markers[z], linestyle="", markersize=14, markerfacecolor="white",
             markeredgecolor="black", label=z) for z in cfg.ZONES]
fig.tight_layout(rect=(0, 0.07, 1, 1))
fig.legend(handles=h, loc="lower center", ncol=7, frameon=True, edgecolor="black", fancybox=False)
save(fig, "performance_diagram")

## 3. Detection by season and climate type

Per-gage scores (event set `EVENT_SET`). A gage needs at least one observed or simulated event in a season.

In [ ]:
det = ctg[(ctg["period_type"] == "season") & (ctg["event_set"] == EVENT_SET)].rename(columns={"period": "season"})
for m in ["CSI", "POD", "FAR"]:
    fig = plots.figure_by_product(det, m, f"{m} ({EVENT_SET} events)", ylim=(0, 1), whis=WHISKERS)
    save(fig, f"season_{m}_by_climate_{EVENT_SET}")

**Larger floods, pooled.** The events of all gages of a climate type are counted together before the score is computed.

In [ ]:
big = ctg[(ctg["period_type"] == "season") & (ctg["event_set"] == "larger")]
pool = pooled(big, ["product", "aridity_zone", "period"]).rename_axis(index={"period": "season"})
pool.round(3).to_csv(FIG / "pooled_detection_larger_by_climate_season.csv")

for m in ["CSI", "POD"]:
    fig, axes = plt.subplots(1, 3, figsize=(20, 7), sharey=True)
    x = np.arange(len(cfg.ZONES))
    for ax, p in zip(axes, P):
        for i, s in enumerate(cfg.SEASONS):
            v = [pool[m].get((p, z, s), np.nan) for z in cfg.ZONES]
            edge, face = plots.SEASON_COLORS[s]
            ax.bar(x + (i - 1.5) * 0.2, v, 0.2, color=face, edgecolor=edge, linewidth=1.8)
        ax.set_xticks(x)
        ax.set_xticklabels(cfg.ZONES, fontweight="bold", fontsize=14)
        ax.set_title(cfg.PRODUCT_LABEL[p])
        ax.grid(axis="y", color="0.88")
        ax.set_axisbelow(True)
    axes[0].set_ylabel(f"Pooled {m} (larger floods)")
    fig.supxlabel("Climate type", fontweight="bold", fontsize=18)
    plots.season_legend_top(fig)
    save(fig, f"season_{m}_pooled_larger")
pool.unstack("season").reindex(columns=cfg.SEASONS, level=1)[["hits", "misses", "false_alarms", "POD", "CSI"]].round(2)

## 4. Maps of detection

Full-period CSI and POD of each gage (event set `EVENT_SET`).

In [ ]:
d = cf[cf["event_set"] == EVENT_SET]
seq = maps.colors_from("YlGnBu", 6, 0.05, 1.0)
fig = maps.figure_metric_maps(d, "CSI", f"CSI ({EVENT_SET} events)", [0.1, 0.2, 0.3, 0.4, 0.5], seq)
save(fig, f"map_CSI_{EVENT_SET}")
fig = maps.figure_metric_maps(d, "POD", f"POD ({EVENT_SET} events)", [0.2, 0.4, 0.6, 0.8], maps.colors_from("YlGnBu", 5, 0.05, 1.0))
save(fig, f"map_POD_{EVENT_SET}")
fig = maps.figure_metric_maps(d, "FAR", f"FAR ({EVENT_SET} events)", [0.2, 0.4, 0.6, 0.8], maps.colors_from("YlOrRd", 5, 0.05, 1.0))
save(fig, f"map_FAR_{EVENT_SET}")

## 5. Matched events by season and climate type

Each gage contributes the median over its matched events of a season.

Peak ratio = simulated peak / observed peak on a logarithmic axis (1 = perfect, 4 = four times too high). Peak timing error = simulated minus observed peak time (negative = early; the matching allows 12 h either way). Time to peak error = simulated minus observed rise time from the event start to the peak (positive = the simulated flood rises more slowly).

In [ ]:
def ratio_axis(ax):
    ax.set_yscale("log")
    ax.set_yticks([1 / 64, 1 / 16, 0.25, 1, 4, 16, 64, 256])
    ax.set_yticklabels(["1/64", "1/16", "1/4", "1", "4", "16", "64", "256"])
    ax.minorticks_off()

ev = ems[(ems["period_type"] == "season") & (ems["event_set"] == EVENT_SET)].rename(columns={"period": "season"})
fig = plots.figure_by_product(ev, "peak_ratio_median", f"Peak ratio, sim / obs ({EVENT_SET} events)", ref=1, whis=WHISKERS)
for ax in fig.axes:
    ratio_axis(ax)
save(fig, f"season_peak_ratio_{EVENT_SET}")
fig = plots.figure_by_product(ev, "time_to_peak_err_h_median", f"Peak timing error (h) ({EVENT_SET} events)",
                              ref=0, ylim=(-12.5, 12.5), whis=WHISKERS)
save(fig, f"season_peak_timing_{EVENT_SET}")

In [ ]:
fig = plots.figure_by_product(ev, "rise_err_h_median", f"Time to peak error (h) ({EVENT_SET} events)", ref=0, whis=WHISKERS)
save(fig, f"season_time_to_peak_{EVENT_SET}")
fig = plots.figure_by_product(ev, "vol_err_pct_median", f"Event volume error (%) ({EVENT_SET} events)", ref=0, whis=WHISKERS)
save(fig, f"season_volume_error_{EVENT_SET}")

In [ ]:
med_ev = (ev.groupby(["product", "aridity_zone", "season"])[["peak_ratio_median", "time_to_peak_err_h_median",
                                                              "rise_err_h_median", "vol_err_pct_median"]].median()
          .round(2).unstack("season").reindex(columns=cfg.SEASONS, level=1)
          .reindex(index=[(p, z) for p in P for z in cfg.ZONES]))
med_ev.to_csv(FIG / f"medians_matched_events_by_climate_season_{EVENT_SET}.csv")
med_ev

## 6. Full period by climate type, the three products side by side

In [ ]:
evf = ems[(ems["period_type"] == "full") & (ems["event_set"] == EVENT_SET)]
fig = plots.figure_zone_product(evf, "peak_ratio_median", f"Peak ratio, sim / obs ({EVENT_SET} events)", ref=1, whis=WHISKERS)
ratio_axis(fig.axes[0])
save(fig, f"full_peak_ratio_by_climate_{EVENT_SET}")
fig = plots.figure_zone_product(evf, "time_to_peak_err_h_median", f"Peak timing error (h) ({EVENT_SET} events)",
                                ref=0, whis=WHISKERS)
save(fig, f"full_peak_timing_by_climate_{EVENT_SET}")
fig = plots.figure_zone_product(cf[cf["event_set"] == EVENT_SET], "CSI", f"CSI ({EVENT_SET} events)", ylim=(0, 1), whis=WHISKERS)
save(fig, f"full_CSI_by_climate_{EVENT_SET}")

## 7. Maps of the matched-event errors

In [ ]:
fig = maps.figure_metric_maps(evf, "peak_ratio_median", f"Median peak ratio, sim / obs ({EVENT_SET} events)",
                              [0.5, 0.8, 1.25, 2, 4, 8],
                              ["#8c510a", "#d8b365", "#ffffff", "#c7eae5", "#5ab4ac", "#01665e", "#003c30"],
                              labels=["below 1/2", "1/2 to 0.8", "0.8 to 1.25", "1.25 to 2", "2 to 4", "4 to 8", "above 8"])
save(fig, f"map_peak_ratio_{EVENT_SET}")
fig = maps.figure_metric_maps(evf, "time_to_peak_err_h_median", f"Median peak timing error (h) ({EVENT_SET} events)",
                              [-6, -3, -1, 1, 3, 6], maps.colors_from("PuOr", 7))
save(fig, f"map_peak_timing_{EVENT_SET}")

## 8. Simulated against observed peaks, every matched event

Unit peak = peak flow divided by the drainage area, so basins of different size share one axis. Darker = more events.

In [ ]:
area = gages.set_index("site")["area_km2"]
e = em[em["event_set"] == EVENT_SET].copy()
e["uo"] = e["peak_obs_cms"] / e["gage"].map(area)
e["us"] = e["peak_sim_cms"] / e["gage"].map(area)
lim = (1e-3, 30)
fig, axes = plt.subplots(1, 3, figsize=(20, 7.4), sharey=True)
for ax, p in zip(axes, P):
    d = e[e["product"] == p]
    hb = ax.hexbin(d["uo"], d["us"], gridsize=55, xscale="log", yscale="log", bins="log", cmap="Greys",
                   extent=(-3, np.log10(30), -3, np.log10(30)), mincnt=1)
    ax.plot(lim, lim, color="#c0392b", lw=2)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_title(f"{cfg.PRODUCT_LABEL[p]} ({len(d):,} events)")
    ax.set_xlabel("Observed unit peak (m3/s per km2)")
    ax.set_aspect("equal")
    ax.grid(color="0.9")
axes[0].set_ylabel("Simulated unit peak (m3/s per km2)")
fig.tight_layout()
save(fig, f"scatter_unit_peaks_{EVENT_SET}")

## Ideas to go further

- Detection against flood size: split `event_matches` by observed unit peak classes and compute POD per class.
- Misses and false alarms have their own tables (`event_matches` with status miss, `false_alarms`); mapping where and when they cluster would show the seasons and regions where radar-only rain loses floods.
- Per-event changes between products: join `event_metrics` of two products on `gage` and `obs_event_id` to see how the same observed flood is simulated with each rain product.
- By basin size: `plots.figure_zone_product(..., group="area_class", groups=cfg.AREA_LABELS, xlabel="Drainage area (km2)")`.